[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rycroft-group/math714/blob/main/i_mixed/mixed.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy.sparse import csr_array
import scipy.sparse.linalg as sla

# Optional: a library for plotting with LaTeX-like
# styles nicer formatted figures
# Warning: need to have LaTeX installed
import scienceplots
plt.style.use(['science'])

# Mixed equations

- [Demo 1: Spinodal decomposition](#spinodal-decomposition)
  - [Setup](#setup-for-spinodal-decomposition)
  - [Assemble the sparse matrices](#assemble-the-sparse-matrices)
  - [Integrate the PDE](#integrate-with-strang-splitting)
  - [Visualize](#visualize-spinodal-decomposition)

<a id="spinodal-decomposition"></a>
## Demo 1: Spinodal decomposition

We will solve the reaction–diffusion equation of Allen–Cahn type
$$
u_t = \epsilon^2 \nabla^2 u + \beta u (1-u^2)
$$
on the periodic unit square $0 \leq x, y < 1$, with $\beta = 3$ and $\epsilon = h = 1/80$, the grid spacing, so that the diffusion coefficient is $\epsilon^2 = h^2$. The initial condition is random, with $u(x,y,0)$ drawn uniformly from $[-1,1]$ at each gridpoint.

The reaction term drives $u$ toward the two stable states $u = \pm 1$, while the diffusion term smooths out sharp gradients. Starting from random noise, the field separates into domains where $u \approx 1$ and $u \approx -1$, which then coarsen over time. This resembles spinodal decomposition in a binary mixture.

The equation mixes two terms with different character. The diffusion term is stiff and is best handled implicitly, while the nonlinear reaction term is cheap to handle explicitly. We therefore use **Strang splitting** over each timestep $k$:

1. Reaction half step: $u \leftarrow u + \frac{k}{2} \beta u (1-u^2)$ (forward Euler over $k/2$),
2. Diffusion full step: Crank–Nicolson over $k$,
3. Reaction half step, as in step 1.

<a id="setup-for-spinodal-decomposition"></a>
### Setup

In [ ]:
# Grid size
m = 80
mm = m*m
h = 1./m
k = h

# Miscellaneous parameters
beta = 3
snaps = 200
iters = 10

# Initial grid
u = np.random.uniform(-1, 1, (m, m))

<a id="assemble-the-sparse-matrices"></a>
### Assemble the sparse matrices

The grid is stored as a vector of length $m^2$, with gridpoint $(i,j)$ at index $l = i + mj$. Crank–Nicolson for the diffusion term requires solving
$$
\left(I - \tfrac{k}{2}\epsilon^2 \Delta_h\right) u^{n+1} = \left(I + \tfrac{k}{2}\epsilon^2 \Delta_h\right) u^{n}
$$
at each step, where $\Delta_h$ is the five-point Laplacian. The implicit matrix $A_i$ and the explicit matrix $A_e$ are assembled directly in compressed sparse row (CSR) format. Each row has five nonzero entries, and the neighbor indices wrap around the edges of the grid to enforce periodicity.

In the code, `fac = 0.5*k/h*h` evaluates left to right to `0.5*k`, so the diffusion coefficient is $\epsilon^2 = h^2$. Equivalently, this is the equation $u_t = \nabla^2 u + \beta u(1-u^2)$ with lengths measured in units of the grid spacing.

In [ ]:
# Create arrays to assemble sparse matrices
iptr = np.empty((mm+1), dtype=np.int32)
idx = np.empty((5*mm), dtype=np.int32)
dat_i = np.empty((5*mm))
dat_e = np.empty((5*mm))
fac = 0.5*k/h*h
cen_i = 1+4*fac
cen_e = 1-4*fac

for j in range(m):
    for i in range(m):
        l = i+m*j
        iptr[l] = 5*l

        # Create list of indices of orthogonal neighbors, taking into account periodicity
        w = [l]
        if j == 0: w += [l-m+mm]
        else: w += [l-m]
        if i == 0: w += [l-1+m]
        else: w += [l-1]
        if i == m-1: w += [l+1-m]
        else: w += [l+1]
        if j == m-1: w += [l+m-mm]
        else: w += [l+m]

        # Sort the list of indices, and add them to the list for assembling the sparse matrices
        w.sort()
        for q in range(5):
            p = 5*l+q
            idx[p] = w[q]
            if w[q] == l:
                dat_i[p] = cen_i
                dat_e[p] = cen_e
            else:
                dat_i[p] = -fac
                dat_e[p] = fac
iptr[mm] = 5*mm
A_i = csr_array((dat_i, idx, iptr), shape=(mm, mm))
A_e = csr_array((dat_e, idx, iptr), shape=(mm, mm))

<a id="integrate-with-strang-splitting"></a>
### Integrate the PDE

The script `spinodal.py` writes each snapshot to the **spinodal.odr** directory in Gnuplot binary matrix format. Here the snapshots are stored in the array `z` instead. The integration takes around a minute.

In [ ]:
# Crank-Nicolson step to integrate the diffusive term over a timestep k
def c_n(u):
    v = A_e.dot(u)
    u[:] = sla.spsolve(A_i, v)

# Explicit step to handle the reaction terms over a timestep k/2
def react(u):
    u[:] += 0.5*beta*k*u*(np.ones(mm)-u*u)

In [ ]:
# Store snapshots of the solution on the grid
z = np.empty((snaps+1, m, m))
z[0] = u.reshape(m, m)

# Perform timesteps
u.shape = (mm)
for i in range(1, snaps+1):
    for j in range(iters):

        # Strang splitting
        react(u)
        c_n(u)
        react(u)

    # Save output snapshot, converting data to a grid
    z[i] = u.reshape(m, m)
    if i % 20 == 0:
        print("# Output frame", i)

<a id="visualize-spinodal-decomposition"></a>
### Visualize

In [ ]:
frames = [0, 2, 10, 40, 100, 200]
fig, axes = plt.subplots(2, 3, figsize=(9, 5.6), dpi=300,
                         sharex=True, sharey=True)

for ax, i in zip(axes.flat, frames):
    im = ax.imshow(z[i], cmap='RdBu_r', vmin=-1, vmax=1,
                   origin='lower', extent=[0, 1, 0, 1])
    ax.set_title(f'$t={i*iters*k:.2f}$')

# Formatting
for ax in axes[1, :]:
    ax.set_xlabel('$x$')
for ax in axes[:, 0]:
    ax.set_ylabel('$y$')
fig.colorbar(im, ax=axes, label='$u(x,y,t)$', shrink=0.8)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.4), dpi=100)
im = ax.imshow(z[0], cmap='RdBu_r', vmin=-1, vmax=1,
               origin='lower', extent=[0, 1, 0, 1])
timestamp = ax.set_title('')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
fig.colorbar(im, ax=ax, label='$u(x,y,t)$')

def animate(i):
    im.set_data(z[i])
    timestamp.set_text(f'$t={i*iters*k:.2f}$')
    return im, timestamp

ani = FuncAnimation(fig, animate, frames=snaps+1, interval=50)
plt.close(fig)
HTML(ani.to_jshtml())